In [1]:
import random
import numpy as np
from PIL import Image

import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.v2 import Compose, Normalize

# Custom imports
%run -i ../fix_path.py
from data_generation.image_classification import generate_dataset
from stepbystep.v0 import StepByStep

## Convolution
**What a convolution is?**

A convolution is an operation where a small matrix (called a filter or kernel) slides over an image.
At each position, it multiplies its values with the pixels underneath (receptive field) and sums them into one output value.
Repeating this across the entire image produces a new transformed image called a feature map.

<div style="display:flex; gap:10px;">
  <img src="../../images/chapter_5-img1.png" style="width:50%;">
  <img src="../../images/chapter_5-img2.png" style="width:50%;">
</div>

In [ ]:
# batch_size=1, 1 channel, 6x6 image
single = np.array([[
        [
            [5, 0, 8, 7, 8, 1],
            [1, 9, 5, 0, 7, 7],
            [6, 0, 2, 4, 6, 6],
            [9, 7, 6, 6, 8, 4],
            [8, 3, 8, 5, 1, 3],
            [7, 2, 7, 0, 1, 0]
        ]
    ]])
single.shape

(1, 1, 6, 6)

In [5]:
identity = np.array([[
    [
        [0, 0, 0],
        [0, 1, 0],
        [0, 0, 0]
    ]
]])

identity.shape

(1, 1, 3, 3)

In [6]:
region = single[:, :, 0:3, 0:3]
filtered_region = region * identity
filtered_region, filtered_region.sum()

(array([[[[0, 0, 0],
          [0, 9, 0],
          [0, 0, 0]]]]),
 np.int64(9))

## Filters/kernels

A filter (or kernel) is a small grid of numbers (like 3×3 or 5×5) that acts as a pattern detector.
Different filters detect different visual patterns, such as:

- edges (horizontal, vertical, diagonal)
- corners
- textures
- smooth regions
- color transitions

Filters start with random numbers and learn useful patterns during training through backpropagation.
<div style="display:flex; gap:10px;">
  <img src="../../images/chapter_5-img3.png" style="width:50%;">
  <img src="../../images/chapter_5-img4.png" style="width:50%;">
</div>

## What convolutions achieve

Convolutions allow a neural network to:

- extract local spatial patterns (edges, shapes, textures)
- build hierarchical understanding (edges → shapes → objects)
- preserve spatial structure (which pixels are near each other)
- reduce parameters dramatically (same filter is reused everywhere)
- detect patterns regardless of where they appear in the image

The result is a set of feature maps that describe what important visual features are present and where.

<div style="display:flex; gap:10px;">
  <img src="../../images/chapter_5-img5.png" style="width:50%;">
  <img src="../../images/chapter_5-img6.png" style="width:50%;">
</div>

## Why we need them

We need convolutions because:

- Images have spatial structure, and treating pixels independently (like flattening) destroys that structure.
- Convolutions are efficient: far fewer parameters than fully connected layers.
- They provide translation invariance — a feature detected in one corner is recognized anywhere.
- They enable deep networks to learn visual concepts from simple local patterns to complex objects.

## How to Determine the Size of the Feature Map given an Input Image
**Without Padding (Valid Convolution)**
$$
\begin{align*}
H_{\text{out}} &= H_{\text{in}} - K_h + 1 \\
W_{\text{out}} &= W_{\text{in}} - K_w + 1
\end{align*}
$$

**With padding $P$ (same convolution or padded convolution)**
$$
\begin{align*}
H_{\text{out}} &= \left( \frac{H_{\text{in}} + 2P - K_h}{S} \right) + 1 \\
W_{\text{out}} &= \left( \frac{W_{\text{in}} + 2P - K_w}{S} \right) + 1
\end{align*}
$$
**Where:**
- $H_{out}$, $W_{out}$ = size of the feature map
- $H_{in}$, $W_{in}$ = input image height and width
- $K_h$, $K_w$ = kernel height and width
- $P$ = padding
- $S$ = stride

**PS: The larger the filter, the smaller the resulting image.**

## Convolving in PyTorch

In [12]:
image = torch.as_tensor(single).float()
kernel_identity = torch.as_tensor(identity).float()
image, kernel_identity

(tensor([[[[5., 0., 8., 7., 8., 1.],
           [1., 9., 5., 0., 7., 7.],
           [6., 0., 2., 4., 6., 6.],
           [9., 7., 6., 6., 8., 4.],
           [8., 3., 8., 5., 1., 3.],
           [7., 2., 7., 0., 1., 0.]]]]),
 tensor([[[[0., 0., 0.],
           [0., 1., 0.],
           [0., 0., 0.]]]]))

Just like the activation functions we saw in Chapter 4, convolutions come in two flavors: functional and module. There is a fundamental difference between the two, though: The functional convolution takes the kernel / filter as an argument while the module has (learnable) weights to represent the kernel / filter.

In [13]:
convolved = F.conv2d(image, kernel_identity, stride=1)
convolved

tensor([[[[9., 5., 0., 7.],
          [0., 2., 4., 6.],
          [7., 6., 6., 8.],
          [3., 8., 5., 1.]]]])

Now, let’s turn our attention to PyTorch’s convolution module, nn.Conv2d. It has many arguments; let’s focus on the first four of them:
- `in_channels`: number of channels of the input image
- `out_channels`: number of channels produced by the convolution
- `kernel_size`: size of the (square) convolution filter / kernel
- `stride`: the size of the movement of the selected region

There are a couple of things to notice here. First, there is no argument for the kernel / filter itself, there is only a kernel_size argument.

The actual filter, that is, the square matrix used to perform element-wise multiplication, is learned by the module.

Second, it is possible to produce multiple channels as output. It simply means the module is going to learn multiple filters. Each filter is going to produce a different
result, which is being called a channel here.
So far, we’ve been using a single-channel image as input, and applying one filter (size three by three) to it, moving one pixel at a time, resulting in one output per
channel. Let’s do it in code:

In [14]:
conv = nn.Conv2d(
in_channels=1, out_channels=1, kernel_size=3, stride=1
)
conv(image)

tensor([[[[ 0.8397, -1.2580,  1.6653,  0.0717],
          [-1.1374,  0.2273,  2.7193, -0.1412],
          [-0.0080,  0.4512, -0.8116, -1.1813],
          [ 0.1192, -0.4856, -1.9434, -1.1408]]]],
       grad_fn=<ConvolutionBackward0>)

## Stride Size 
The number of pixels to move the filter as we apply it on the image. So far we have been using a stride of 1, now we'll try out a stride of size 2.

<div style="display:flex; gap:10px;">
  <img src="../../images/chapter_5-img7.png" style="width:50%;">
  <img src="../../images/chapter_5-img8.png" style="width:50%;">
</div>


In [15]:
convolved_stride2 = F.conv2d(image, kernel_identity, stride=2)
convolved_stride2

tensor([[[[9., 0.],
          [7., 6.]]]])

## Padding
Notice that applying a filter to an image shrinks it (as can be seen in the result of the feature map). What if we wanted to maintain the shape of the original image? To do that, we use padding where we sorround the image with zeros effectively increasing it's size and maintaining it's shape after convolution.

See? By adding columns and rows of zeros around it, we expand the input image such that the gray region starts centered in the actual top left corner of the input image. This simple trick can be used to preserve the original size of the image.

![chapter_5-img9](../../images/chapter_5-img9.png)

In code, as usual, PyTorch gives us two options: functional (F.pad()) and module (nn.ConstantPad2d). Let’s start with the module version this time:

In [16]:
constant_padder = nn.ConstantPad2d(padding=1, value=0)
constant_padder(image)

tensor([[[[0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 5., 0., 8., 7., 8., 1., 0.],
          [0., 1., 9., 5., 0., 7., 7., 0.],
          [0., 6., 0., 2., 4., 6., 6., 0.],
          [0., 9., 7., 6., 6., 8., 4., 0.],
          [0., 8., 3., 8., 5., 1., 3., 0.],
          [0., 7., 2., 7., 0., 1., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.]]]])

There are two arguments: padding, for the number of columns and rows to be stuffed in the image; and value, for the value that is filling these new columns and rows. One can also do asymmetric padding by specifying a tuple in the padding argument representing (left, right, top, bottom). So, if we were to stuff our image on the left and right sides only, the argument would go like this: (1, 1, 0, 0). 

**In the functional version, one must specify the padding as a tuple.**

We can achieve the same result using the functional padding:

In [17]:
padded = F.pad(image, pad=(1, 1, 1, 1), mode='constant', value=0)
padded

tensor([[[[0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 5., 0., 8., 7., 8., 1., 0.],
          [0., 1., 9., 5., 0., 7., 7., 0.],
          [0., 6., 0., 2., 4., 6., 6., 0.],
          [0., 9., 7., 6., 6., 8., 4., 0.],
          [0., 8., 3., 8., 5., 1., 3., 0.],
          [0., 7., 2., 7., 0., 1., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.]]]])